Implement a custom standardization and text vectorization pipeline that compares word-level tokenization with an OOV token against character-level/subword-adjacent preprocessing.

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [2]:
# Sample corpus (simulating training data)
train_corpus = [
    "Deep learning empowers speech recognition systems.",
    "Natural language processing bridges human and machine communication.",
    "TensorFlow and Keras make prototyping fast."
]

In [3]:
# Evaluation data containing unseen words (OOV)
test_corpus = [
    "Deep learning supercharges automated speech systems.", # 'supercharges', 'automated' are OOV
    "PyTorch and JAX are alternative frameworks." # 'PyTorch', 'JAX', 'alternative', 'frameworks' are OOV
]

In [4]:
# 1. Configure Word-level Vectorization with an OOV token
word_vectorizer = TextVectorization(
    max_tokens=20,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=10
)

word_vectorizer.adapt(train_corpus)

In [5]:
# 2. Configure a Character-level Vectorization as an alternative to mitigate OOV
char_vectorizer = TextVectorization(
    standardize="lower_and_strip_punctuation",
    split="character",
    output_mode="int",
    output_sequence_length=40
)

char_vectorizer.adapt(train_corpus)

In [6]:

# Process test data
word_encoded = word_vectorizer(test_corpus)

char_encoded = char_vectorizer(test_corpus)

print("Vocabulary:", word_vectorizer.get_vocabulary())
print("\nWord-level Encoded Test Matrix:\n", word_encoded.numpy())

print("\nChar-level Encoded Test Matrix Shape:", char_encoded.shape)


Vocabulary: ['', '[UNK]', np.str_('and'), np.str_('tensorflow'), np.str_('systems'), np.str_('speech'), np.str_('recognition'), np.str_('prototyping'), np.str_('processing'), np.str_('natural'), np.str_('make'), np.str_('machine'), np.str_('learning'), np.str_('language'), np.str_('keras'), np.str_('human'), np.str_('fast'), np.str_('empowers'), np.str_('deep'), np.str_('communication')]

Word-level Encoded Test Matrix:
 [[18 12  1  1  5  4  0  0  0  0]
 [ 1  2  1  1  1  1  0  0  0  0]]

Char-level Encoded Test Matrix Shape: (2, 40)


Task 1:

Look at the word_encoded matrix. How many times did the [UNK] token (index 1) appear? What vital semantic information is lost during down-stream tasks when a model encounters a high ratio of OOV tokens?

In [8]:
unk_token_count = tf.reduce_sum(tf.cast(tf.equal(word_encoded, 1), tf.int32))
print(f"The [UNK] token (index 1) appeared {unk_token_count.numpy()} times.")

# Semantic information lost due to a high ratio of OOV tokens:
# vital semantic information is lost because:
# 1. Contextual understanding is diminished: OOV tokens are replaced by a generic [UNK] token,
#    which strips away the specific meaning of the original word. This makes it harder for the model
#    to understand the full context and nuance of a sentence or document.
# 2. Reduced model performance: Downstream tasks like sentiment analysis, machine translation,
#    or question answering rely heavily on the precise meaning of words. Replacing many words
#    with [UNK] degrades the input quality, leading to poorer performance.
# 3. Inability to generalize to new vocabulary: If a significant portion of new text consists of OOV words,
#    the model cannot effectively process or learn from this new information.
# 4. Bias introduction: If OOV tokens are more prevalent in certain types of text or domains,
#    the model might perform worse on those specific inputs, introducing bias.

The [UNK] token (index 1) appeared 7 times.


Task 2:

Character-level tokenization completely eliminates the OOV problem, but what severe trade-offs does it introduce regarding sequence length, computational complexity (Big O), and the model's ability to capture long-range dependencies?

Character-level tokenization effectively eliminates the OOV problem because all words can be composed from a finite set of characters. However, it introduces several significant trade-offs:

1.  **Sequence Length:**
    *   **Trade-off**: Character-level sequences are significantly longer than word-level sequences for the same text. Each word, which might be a single token in word-level tokenization, becomes multiple tokens (characters) in character-level tokenization.
    *   **Impact**: Longer sequences require more memory and computational resources (especially for models like Transformers or LSTMs that process sequences sequentially or quadratically with respect to length).

2.  **Computational Complexity (Big O):**
    *   **Trade-off**: Models often have computational complexities that scale with the sequence length, sometimes quadratically (e.g., $O(L^2)$ for self-attention mechanisms in Transformers) or linearly (e.g., $O(L)$ for RNNs).
    *   **Impact**: Due to much longer sequences, character-level models will experience a substantial increase in computational cost and training time compared to word-level models. For example, a quadratic complexity model would be significantly slower on a character sequence that is 5-10 times longer than its word-level counterpart.

3.  **Model's Ability to Capture Long-Range Dependencies:**
    *   **Trade-off**: While character-level models *can* capture long-range dependencies, it becomes more challenging and less efficient. To understand the meaning of a word, the model needs to process multiple characters that form that word. To understand a sentence, it then needs to process many more character tokens across the entire sentence.
    *   **Impact**: The 'distance' between semantically related units (words or phrases) in the input sequence becomes much larger at the character level. This can make it harder for the model to effectively learn and maintain dependencies between distant but semantically related parts of the text, potentially requiring deeper networks or more attention heads to compensate, further increasing computational burden and data requirements.